# OWL 2 Manchester Syntax — User Guide

Manchester Syntax is a frame-based, English-readable notation for OWL 2 ontologies — `Class: Cat SubClassOf: Animal` rather than raw RDF triples. StarLayer parses it natively via `format="manchester"` (alias `"omn"`) — no Java or OWL API dependency — mapping every frame and axiom into ordinary RDF triples per the OWL 2 Mapping to RDF Graphs specification.

## How to run this notebook

This guide assumes StarLayer has been pip installed. No extra runtime dependencies are needed — Manchester Syntax parsing is pure Python.

Run cells from top to bottom.

In [1]:
from starlayer import StarLayerGraph, Namespace
from rdflib.namespace import OWL

EX = Namespace("http://example.org/vehicles#")


def show(manchester_text):
    """Parse a Manchester Syntax snippet and print the RDF it maps to."""
    g = StarLayerGraph()
    g.bind("", EX)
    g.bind("owl", OWL)
    g.parse(data=f"Prefix: : <{EX}>\n" + manchester_text, format="manchester")
    print(g.serialize(format="turtle12"))
    return g

## 1. Classes: `SubClassOf:`, `SuperClassOf:`, `DisjointWith:`

Each `Class:` frame declares an `owl:Class`; its clauses add axioms about it. `SuperClassOf:` is the reverse-direction spelling of `SubClassOf:` — `Vehicle SuperClassOf: Motorcycle` means exactly the same thing as `Motorcycle SubClassOf: Vehicle`.

In [2]:
_ = show("""
Class: Vehicle
Class: Car
    SubClassOf: Vehicle
Class: Motorcycle
    SubClassOf: Vehicle
Class: Truck
    SubClassOf: Vehicle
    DisjointWith: Car

Class: Vehicle
    SuperClassOf: Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Motorcycle a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Truck a owl:Class ;
    rdfs:subClassOf :Vehicle ;
    owl:disjointWith :Car .

:Vehicle a owl:Class .



## 2. Class expressions: `and` / `or` / `not` / `that`

`that` is a readability synonym for `and`, used for refining a restriction in a sentence-like way ("a family vehicle that has airbags").

In [3]:
_ = show("""
Class: Car
Class: Truck
DataProperty: hasAirbags

Class: ElectricCar
    SubClassOf: Car and (not Truck)

Class: FamilyVehicle
    EquivalentTo: Car or Truck

Class: SafeFamilyVehicle
    EquivalentTo: FamilyVehicle that (hasAirbags value true)
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Car a owl:Class .

:ElectricCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Class ;
        owl:intersectionOf ( :Car [
            a owl:Class ;
            owl:complementOf :Truck
        ] )
    ] .

:FamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:unionOf ( :Car :Truck )
    ] .

:SafeFamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf ( :FamilyVehicle [
            a owl:Restriction ;
            owl:hasValue true ;
            owl:onProperty :hasAirbags
        ] )
    ] .

:Truck a owl:Class .

:hasAirbags a owl:DatatypeProperty .



## 3. Restrictions: `some` / `only` / `onlysome` / `value` / `Self`, and cardinality

`onlysome` is shorthand for `(p some C) and (p only C)` together. `min`/`max`/`exactly` work with or without a qualifying filler class — compare `SmallCar` (qualified) below to `BigCar` (unqualified).

In [4]:
_ = show("""
Class: Car
Class: Person
ObjectProperty: hasPassenger
ObjectProperty: hasOwner
Individual: Alice
ObjectProperty: monitorsItself
DataProperty: hasDoors

Class: FullCar
    SubClassOf: Car
    SubClassOf: hasPassenger some Person

Class: PassengerOnlyCar
    SubClassOf: Car
    SubClassOf: hasPassenger only Person

Class: BalancedCar
    SubClassOf: Car
    SubClassOf: hasPassenger onlysome Person

Class: AlicesCar
    SubClassOf: Car
    SubClassOf: hasOwner value Alice

Class: SelfMonitoringCar
    SubClassOf: Car
    SubClassOf: monitorsItself Self

Class: Sedan
    SubClassOf: Car
    SubClassOf: hasDoors exactly 4

Class: SmallCar
    SubClassOf: Car
    SubClassOf: hasPassenger max 2 Person

Class: BigCar
    SubClassOf: Car
    SubClassOf: hasPassenger min 5 Person
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Alice a owl:NamedIndividual .

:AlicesCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:hasValue :Alice ;
        owl:onProperty :hasOwner
    ] .

:BalancedCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Class ;
        owl:intersectionOf ( [
            a owl:Restriction ;
            owl:onProperty :hasPassenger ;
            owl:someValuesFrom :Person
        ] [
            a owl:Restriction ;
            owl:allValuesFrom :Person ;
            owl:onProperty :hasPassenger
        ] )
    ], :Car .

:BigCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Restriction ;
        owl:minQualifiedCardinality "5"^^xsd:nonNegativeInteger ;
        owl:onClass :Person ;
        owl:onProperty :hasPassenger
    ], :Car .

:Car a owl:Class .

:FullCar

## 4. Enumerations: `{a, b, c}`

A class defined as exactly the individuals listed — `owl:oneOf`.

In [5]:
_ = show("""
Individual: Red
Individual: Green
Individual: Blue

Class: PrimaryColor
    EquivalentTo: {Red, Green, Blue}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:Blue a owl:NamedIndividual .

:Green a owl:NamedIndividual .

:PrimaryColor a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:oneOf ( :Red :Green :Blue )
    ] .

:Red a owl:NamedIndividual .



## 5. Object and data properties

`Domain:`/`Range:`, `Characteristics:`, `InverseOf:`, `SubPropertyOf:`/`SuperPropertyOf:`, and `SubPropertyChain:` (object properties only — `hasParent o hasParent` composed into `hasGrandparent`).

In [6]:
_ = show("""
Class: Vehicle
Class: Engine
ObjectProperty: hasEngine
    Domain: Vehicle
    Range: Engine
    Characteristics: Functional

ObjectProperty: poweredBy
    InverseOf: hasEngine

ObjectProperty: hasSubEngine
    SubPropertyOf: hasEngine

ObjectProperty: hasComponent
    SuperPropertyOf: hasSubEngine

ObjectProperty: hasParent
ObjectProperty: hasGrandparent
    SubPropertyChain: hasParent o hasParent

DataProperty: hasWeightKg
    Domain: Vehicle
    Range: xsd:decimal
    Characteristics: Functional
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Engine a owl:Class .

:Vehicle a owl:Class .

:hasComponent a owl:ObjectProperty .

:hasEngine a owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range :Engine .

:hasGrandparent a owl:ObjectProperty ;
    owl:propertyChainAxiom ( :hasParent :hasParent ) .

:hasParent a owl:ObjectProperty .

:hasSubEngine a owl:ObjectProperty ;
    rdfs:subPropertyOf :hasComponent, :hasEngine .

:hasWeightKg a owl:DatatypeProperty, owl:FunctionalProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range xsd:decimal .

:poweredBy a owl:ObjectProperty ;
    owl:inverseOf :hasEngine .



## 6. Individuals: `Types:`, `Facts:`, `SameAs:`, `DifferentFrom:`

`Facts:` assertions can be negated with a leading `not` — mapped to `owl:NegativePropertyAssertion`, not silently dropped.

In [7]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
DataProperty: hasVIN
Individual: Alice
Individual: Bob
Individual: mycar2

Individual: mycar
    Types: Car
    Facts: hasOwner Alice, hasVIN "1HGCM82633A004352", not hasOwner Bob
    SameAs: mycar2
    DifferentFrom: Bob
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:Alice a owl:NamedIndividual .

:Bob a owl:NamedIndividual .

:Car a owl:Class .

:hasOwner a owl:ObjectProperty .

:hasVIN a owl:DatatypeProperty .

:mycar :hasOwner :Alice ;
    :hasVIN "1HGCM82633A004352" ;
    a owl:NamedIndividual, :Car ;
    owl:differentFrom :Bob ;
    owl:sameAs :mycar2 .

:mycar2 a owl:NamedIndividual .

_:N6831271cc7b9435aa7be97d1347594cc a owl:NegativePropertyAssertion ;
    owl:assertionProperty :hasOwner ;
    owl:sourceIndividual :mycar ;
    owl:targetIndividual :Bob .



## 7. Data ranges and `Datatype:` frames

Facet-restricted datatypes (`xsd:integer[>= 0, <= 400]`), a union of datatypes, and an explicit enumeration of data values (`DataOneOf`).

In [8]:
_ = show("""
Datatype: PositiveInt
    EquivalentTo: xsd:integer[>= 0]

DataProperty: hasSpeedKmh
    Range: xsd:integer[>= 0, <= 400]

DataProperty: hasFuelType
    Range: xsd:string or xsd:integer

DataProperty: hasStatusCode
    Range: {"active", "retired"}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:PositiveInt a rdfs:Datatype ;
    owl:equivalentClass [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions ( [
            xsd:minInclusive 0
        ] )
    ] .

:hasFuelType a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:unionOf ( xsd:string xsd:integer )
    ] .

:hasSpeedKmh a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions ( [
            xsd:minInclusive 0
        ] [
            xsd:maxInclusive 400
        ] )
    ] .

:hasStatusCode a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:oneOf ( "active" "retired" )
    ] .



## 8. `HasKey:` and `DisjointUnionOf:`

`HasKey:` says a property (or set of properties) uniquely identifies individuals of a class; `DisjointUnionOf:` says a class is exactly the (pairwise disjoint) union of the listed subclasses.

In [9]:
_ = show("""
Class: Car
Class: Truck
Class: Motorcycle
DataProperty: hasVIN

Class: Vehicle
    HasKey: hasVIN
    DisjointUnionOf: Car, Truck, Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:Car a owl:Class .

:Motorcycle a owl:Class .

:Truck a owl:Class .

:Vehicle a owl:Class ;
    owl:disjointUnionOf ( :Car :Truck :Motorcycle ) ;
    owl:hasKey ( :hasVIN ) .

:hasVIN a owl:DatatypeProperty .



## 9. Frame-level `Annotations:`

A standalone `Annotations:` clause maps straight to plain triples about the frame's own subject — `rdfs:label`, `rdfs:comment`, or any other annotation property.

In [10]:
_ = show("""
Class: Car
    Annotations: rdfs:label "Car", rdfs:comment "A wheeled motor vehicle"
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:comment "A wheeled motor vehicle" ;
    rdfs:label "Car" .



### 9.a Inline per-axiom `Annotations:`

A different, more targeted form: an `Annotations:` block written *inside* another clause's list, immediately before one specific item, annotates just that one axiom — via `owl:Axiom` reification (`owl:annotatedSource`/`owl:annotatedProperty`/`owl:annotatedTarget` plus the annotation itself), alongside the plain axiom triple, never instead of it. The annotation applies to exactly the item it precedes — `Pet` below is asserted as a superclass of `Cat` too, but isn't annotated.

In [11]:
_ = show("""
Class: Animal
Class: Pet
Class: Cat
    SubClassOf: Annotations: rdfs:comment "domesticated" Animal, Pet
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:subClassOf :Pet, :Animal .

:Pet a owl:Class .

_:N3167b69119a34703bf882e8047b8653e a owl:Axiom ;
    rdfs:comment "domesticated" ;
    owl:annotatedProperty rdfs:subClassOf ;
    owl:annotatedSource :Cat ;
    owl:annotatedTarget :Animal .



## 10. Top-level Misc axioms

`EquivalentClasses:`/`DisjointClasses:` and their property-flavored counterparts `EquivalentProperties:`/`DisjointProperties:`, plus `SameIndividual:`/`DifferentIndividuals:` — each accepts two or more names at once, not just a pair.

In [12]:
_ = show("""
Class: Auto
Class: Car
Class: Motorcar
EquivalentClasses: Auto, Car, Motorcar

Class: Motorcycle
Class: Bicycle
DisjointClasses: Car, Motorcycle, Bicycle

ObjectProperty: hasOwner
ObjectProperty: ownedBy
EquivalentProperties: hasOwner, ownedBy

ObjectProperty: hasPart
DisjointProperties: hasOwner, hasPart

Individual: mycar
Individual: myauto
SameIndividual: mycar, myauto

Individual: yourcar
DifferentIndividuals: mycar, yourcar
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

:Auto a owl:Class ;
    owl:equivalentClass :Car .

:Bicycle a owl:Class .

:Car a owl:Class ;
    owl:equivalentClass :Motorcar .

:Motorcar a owl:Class .

:Motorcycle a owl:Class .

:hasOwner a owl:ObjectProperty ;
    owl:equivalentProperty :ownedBy ;
    owl:propertyDisjointWith :hasPart .

:hasPart a owl:ObjectProperty .

:myauto a owl:NamedIndividual .

:mycar a owl:NamedIndividual ;
    owl:differentFrom :yourcar ;
    owl:sameAs :myauto .

:ownedBy a owl:ObjectProperty .

:yourcar a owl:NamedIndividual .

_:Nc09c186e74c84d19a382532a6230f933 a owl:AllDisjointClasses ;
    owl:members ( :Car :Motorcycle :Bicycle ) .



## 11. Going back: RDF → Manchester Syntax

`format="manchester"` works for `serialize()` too - the reverse direction. This is a genuinely different, harder kind of problem than parsing: rather than deterministically producing RDF from text, the serializer has to *recognize* patterns in a flat, unordered triple set (which blank nodes are restrictions, which are `rdf:List` cells, which came from an `owl:Axiom` annotation block) and reconstruct frames from `rdf:type` declarations. `SuperClassOf:`/`onlysome`/small top-level `EquivalentClasses:`-style Misc axioms always come back in their canonical expanded form (see `packages/graph/docs/manchester_syntax_gap_analysis.md` for exactly why) - everything else round-trips losslessly.

In [13]:
g = StarLayerGraph()
g.bind("", EX)
g.parse(data=f"Prefix: : <{EX}>\n" + """
Class: Animal
Class: Cat
    SubClassOf: Animal
    SubClassOf: hasLegs exactly 4
    Annotations: rdfs:label "Cat"
ObjectProperty: hasLegs
""", format="manchester")

print(g.serialize(format="manchester"))

Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :Animal, :hasLegs exactly 4
    Annotations: rdfs:label "Cat"

ObjectProperty: :hasLegs



## What's not supported yet

A handful of constructs are deliberately unimplemented — they raise `ManchesterSyntaxError` rather than being silently dropped or mis-parsed: SWRL `Rule:`, and inline `Annotations:` specifically on the six top-level Misc axioms from section 10 (`EquivalentClasses:` and friends) — unlike every other clause, their pairwise-chain/`AllDisjoint*` encoding has no single item that a per-item annotation could reify against. Full conformance tracking — including what's matched, what's not, and why — lives in `packages/graph/docs/manchester_syntax_gap_analysis.md`.

In [14]:
from starlayergraph.parsers.errors import ManchesterSyntaxError

try:
    show("""
    Rule: Person(?x)
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")

try:
    show("""
    Class: A
    Class: B
    Class: C
    EquivalentClasses: Annotations: rdfs:comment "why" A, B, C
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")

Rejected as expected: expected a Prefix:/Ontology:/frame/Misc-axiom keyword, got 'Rule:'
Rejected as expected: unknown prefix 'Annotations' - no matching Prefix: directive (the default ":" prefix must be declared explicitly too; it isn't guessed from the ontology IRI)
